# Supplementary Table S8: Multinomial logit regression for the cumulative network till 2017

This notebook fits the multinomial logit models (1)-(6) on the cumulative citation preference network till 2017 (each country pair keeps its most recent significant citation preference) and prints the LaTeX regression table. The models are cross-sectional: one row per country pair with the covariates of 2017.

The 2017 covariates are selected before merging with the cumulative edges. As in the main analysis, the country pairs with at least 50 cited articles are kept. Unlike Supplementary Table S6, the models have no year fixed effects.

In [1]:
import os
from itertools import chain
import pandas as pd
import numpy as np

import statsmodels.api as sm

from vizutils import make_multinomial_latex_table
from sourcedatautils import load_country_auc, load_net_auc, cumulative_network, write_source_data, path2sourcedata, path2data, eyear

# Cumulative citation preference network

In [2]:
# the significant citation preferences (N >= 50, Holm-Sidak correction with alpha = 0.01 within each year)
country_auc_df = load_country_auc()
net_auc = load_net_auc(country_auc_df)

# each country pair keeps its most recent significant citation preference
cumulative_net = cumulative_network(net_auc)
print(cumulative_net.shape[0], 'edges in the cumulative network')

2007 edges in the cumulative network


# Multinomial Link Prediction

In [3]:
panel_data = pd.read_csv(os.path.join(path2data, 'link_prediction_panel_data_1990_2017.csv.gz'))

# cross-section: the covariates of 2017, one row per country pair
panel_2017 = panel_data[panel_data['year'] == eyear].reset_index(drop=True)

# each country pair gets the sign of its cumulative edge (from the year it was last significant)
net_panel = panel_2017.merge(cumulative_net.rename(columns={'CitingCountry':'oCountry', 'CitedCountry':'tCountry'}),
                           how='left', on=['oCountry','tCountry'])

net_panel['edge_preference'] = 1 + (net_panel['Zscore'] < 0).astype(int)
net_panel.loc[net_panel['Significant'].isnull(), 'edge_preference'] = 0

v1=['oGDP_PCAP','tGDP_PCAP', 'oPop','tPop']
v2=['PhysicalDistance','SameContinent']
v3=['comlang_off','TopicSimilarity']
v4=['cumulative_laws']
v5=['CollabStrength']
v6=['oTopFrac','tTopFrac']

all_model_features = v1 + v2 + v3 + v4 + v5 + v6

# continuous variables are standardized once over the 2017 cross-section, so all models share the same scale
for c in all_model_features:
    if net_panel[c].nunique() > 2:
        net_panel[c] = (net_panel[c] - net_panel[c].mean())/net_panel[c].std()

exog_var_sets = [v1,v2,v3,v4,v5,v6]
dep_var = "edge_preference"

Nmodels = len(exog_var_sets)

fit_models = []
for icol in range(Nmodels):
    exog_vars = list(chain.from_iterable(exog_var_sets[:(icol+1)]))
    X = net_panel[exog_vars + [dep_var]].dropna().reset_index(drop=True)

    Y = X[dep_var]
    X = sm.add_constant(X[exog_vars])

    model = sm.MNLogit(Y, X).fit()
    fit_models.append(model)

Optimization terminated successfully.
         Current function value: 0.136701
         Iterations 10
Optimization terminated successfully.
         Current function value: 0.138972
         Iterations 10
Optimization terminated successfully.
         Current function value: 0.131391
         Iterations 11
Optimization terminated successfully.
         Current function value: 0.131244
         Iterations 11
Optimization terminated successfully.
         Current function value: 0.121739
         Iterations 11
Optimization terminated successfully.
         Current function value: 0.115404
         Iterations 11


In [4]:
namedict = {'const':'Intercept', 'oTopFrac':'Origin top journal fraction', 'tTopFrac':'Target top journal fraction',
'oGDP_PCAP':'Log origin GDP per capita', 'oPop':'Log origin population','tPop':'Log target population',
           'tGDP_PCAP':'Log target GDP per capita', 'PhysicalDistance': 'Physical distance', 'TopicSimilarity':'Field similarity',
            'comlang_off':'Same official language', 'CollabStrength':'Log collaboration strength','SameContinent':'Same continent',
           'cumulative_laws':'Bilateral research agreements'}

# the LaTeX tables are saved in the tables folder
path2tables = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'tables')
os.makedirs(path2tables, exist_ok=True)

make_multinomial_latex_table(fit_models, exog_var_sets, dep_var, namedict,
                             caption_text="Multinomial logit regression for the cumulative network till 2017.",
                             label="table:multinomial2017", fname=os.path.join(path2tables, "TableS8.tex"))

{\tiny
    \begin{longtable}{p{0.2\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}}\caption{\textbf{Multinomial logit regression for the cumulative network till 2017.} Model coefficients (log-odds of a positive or negative citation preference relative to no preference) with 95\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The LLR $\chi^2$ is a likelihood-ratio test against the intercept-only model.} 
      \label{table:multinomial2017} \\
      \hline \hline \\
    \multicolumn{7}{c}{\textbf{Dependent variable: Citation preference}} \\ \hline 
     & \multicolumn{6}{c}{Model}  \\\cline{2-7}& (1)& (2)& (3)& (4)& (5)& (6)\\[0.8ex]
    \hline
    \endfirsthead\multicolumn{2}{c}%
      {{\tablename\ \thetable{} -- continued from previous page}} \\
      \hline \

'{\\tiny\n    \\begin{longtable}{p{0.2\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}}\\caption{\\textbf{Multinomial logit regression for the cumulative network till 2017.} Model coefficients (log-odds of a positive or negative citation preference relative to no preference) with 95\\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The LLR $\\chi^2$ is a likelihood-ratio test against the intercept-only model.} \n      \\label{table:multinomial2017} \\\\\n      \\hline \\hline \\\\\n    \\multicolumn{7}{c}{\\textbf{Dependent variable: Citation preference}} \\\\ \\hline \n     & \\multicolumn{6}{c}{Model}  \\\\\\cline{2-7}& (1)& (2)& (3)& (4)& (5)& (6)\\\\[0.8ex]\n    \\hline\n    \\endfirsthead\\multicolumn{2}{c}%\n      {{\\tablename\\ \\thetable{} -- co

The same table split into two tables that each fit on one page: TableS8a (positive citation preference) and TableS8b (negative citation preference and the model statistics), written to `tables/TableS8_split.tex`.

In [5]:
from vizutils import make_multinomial_latex_tables_split

make_multinomial_latex_tables_split(fit_models, exog_var_sets, dep_var, namedict,
                                    caption_text="Multinomial logit regression for the cumulative network till 2017.",
                                    label="table:multinomial2017", fname=os.path.join(path2tables, "TableS8_split.tex"))

\refstepcounter{table}\label{table:multinomial2017}\addtocounter{table}{-1}%
\begin{table}[H]
\let\origthetable\thetable\renewcommand{\thetable}{\origthetable a}
\caption{\textbf{Multinomial logit regression for the cumulative network till 2017. (a) Positive citation preference.} Model coefficients (log-odds of a positive citation preference relative to no preference) with 95\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The coefficients for the negative citation preference and the model statistics are in Table \ref{table:multinomial2017:negative}.}
\label{table:multinomial2017:positive}
\centering
{\tiny
\begin{tabular}{p{0.2\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}}
\hline \hline \\[-1.8ex]
\multicolumn{7}{c}{\textbf{Dependent variable: Citation pre

'\\refstepcounter{table}\\label{table:multinomial2017}\\addtocounter{table}{-1}%\n\\begin{table}[H]\n\\let\\origthetable\\thetable\\renewcommand{\\thetable}{\\origthetable a}\n\\caption{\\textbf{Multinomial logit regression for the cumulative network till 2017. (a) Positive citation preference.} Model coefficients (log-odds of a positive citation preference relative to no preference) with 95\\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The coefficients for the negative citation preference and the model statistics are in Table \\ref{table:multinomial2017:negative}.}\n\\label{table:multinomial2017:positive}\n\\centering\n{\\tiny\n\\begin{tabular}{p{0.2\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}}\n\\hline \\hline \\\\[-1.8ex]\n\\multicolumn{7}{c}{

## Source Data for Supplementary Table 8

The values reported in the table, written to `Source Data/SourceData_8_TableS8.xlsx`.

In [6]:
title = 'Supplementary Table 8: Multinomial logit regression for the cumulative network till 2017.'
description = ('Coefficients of the multinomial logit models (1)-(6) predicting a positive or negative citation '
               'preference in the cumulative network till 2017 (each country pair keeps its most recent significant '
               'citation preference) compared to no preference (baseline), with one row per country pair and the '
               'covariates of 2017. Continuous variables are standardized. '
               'The models have no year fixed effects.')

coefs = []
fits = []
for imodel, model in enumerate(fit_models):
    conf = model.conf_int()
    for icat, outcome in [(1, 'Positive'), (2, 'Negative')]:
        df = pd.DataFrame({'Coefficient':model.params[icat-1], 'SE':model.bse[icat-1], 'p-value':model.pvalues[icat-1]})
        df['CI lower (95%)'] = conf.loc[str(icat)]['lower']
        df['CI upper (95%)'] = conf.loc[str(icat)]['upper']
        df.insert(0, 'Variable', df.index.map(lambda v: namedict.get(v, v)))
        df.insert(0, 'Citation preference', outcome)
        df.insert(0, 'Model', imodel + 1)
        coefs.append(df[['Model', 'Citation preference', 'Variable', 'Coefficient', 'CI lower (95%)', 'CI upper (95%)',
                         'SE', 'p-value']])
    fits.append([imodel + 1, int(model.nobs), model.prsquared, model.llf, model.llr, model.df_model, model.llr_pvalue])

coefs = pd.concat(coefs, ignore_index=True)
fits = pd.DataFrame(fits, columns=['Model', 'Observations', 'Pseudo R2', 'Log likelihood', 'LLR chi2', 'd.f.',
                                   'LLR p-value'])

tables = [('Coefficients', 'Coefficients of each model for the positive and negative citation preference.', coefs),
          ('Model fit', 'Observations and goodness of fit of each model.', fits)]

write_source_data('SourceData_8_TableS8.xlsx', title, description, tables)

wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_8_TableS8.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
